# 🎭 絵文字パーソナリティ・マトリックス

**遊びの実験**: 各モデルに「あなた自身」と「他の 3 モデル」を **それぞれ絵文字 5 つ** で表現してもらう。
4×4 マトリックスを作り、自己認識と他者認識のズレを眺める。

**観察ポイント**:
- 🪞 自己描写は控えめか派手か
- 👀 他モデルへの描写は名前から連想されたステレオタイプか
- 🎨 4 モデルで一致する絵文字 (集合知) はあるか
- 🤔 意外な絵文字を選ぶモデルはどれか


## 1. セットアップ

In [ ]:
import os, re, random
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(
    base_url="https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1",
    api_key=os.environ.get("LLMJP_API_KEY", "dummy"),
    timeout=300.0,
)

def chat(model, prompt, system="日本語で。", max_tokens=3000, temperature=0.8):
    """内部 streaming 1 ショット。
    - thinking モデルでも reasoning を落とさない
    - ストリーミング途中で接続が切れても、それまでの蓄積を返す
      (LLM-jp 8b thinking で MidStreamFallbackError がたまに発生するため)
    """
    sys_msg = system + "\n\n/no_think"
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt}]
    extra = {"chat_template_kwargs": {"enable_thinking": False}}
    def _try(use_extra):
        kwargs = dict(model=model, messages=msgs, max_tokens=max_tokens,
                      temperature=temperature, stream=True)
        if use_extra:
            kwargs["extra_body"] = extra
        return client.chat.completions.create(**kwargs)
    try:
        stream = _try(True)
    except Exception:
        stream = _try(False)
    content, reasoning = [], []
    try:
        for chunk in stream:
            if not chunk.choices: continue
            d = chunk.choices[0].delta
            c = getattr(d, "content", None)
            if c: content.append(c)
            for f in ("reasoning_content", "reasoning"):
                v = getattr(d, f, None)
                if v: reasoning.append(v); break
    except Exception as e:
        print(f"  ⚠️  stream interrupted ({type(e).__name__}); using partial output")
    text = "".join(content).strip()
    return text if text else "".join(reasoning).strip()

ALL = [m.id for m in client.models.list().data]
def pick(s):
    for m in ALL:
        if s.lower() in m.lower(): return m
    raise RuntimeError(f"no model matching {s!r}")

VOICES = {
    "🌸 LLM-jp 8b":  pick("llm-jp-4-8b"),
    "🗻 LLM-jp 32b": pick("llm-jp-4-32b"),
    "🐉 Qwen 27b":   pick("qwen"),
    "💎 Gemma 31b":  pick("gemma"),
}
print("4 モデル準備完了:")
for n, m in VOICES.items():
    print(f"  {n:18s} → {m}")


## 2. 各モデルに 4 モデル分の絵文字を依頼

In [ ]:
# 各モデルに「以下 4 モデル (自分含む) を、それぞれ絵文字 5 つで表現せよ」と一括で頼む
# (4 モデル × 1 呼び出し = 4 呼び出しで matrix が埋まる)

MODEL_HINTS = """\
- LLM-jp 8b: NII が開発した 80 億パラメータの日本語特化モデル。比較的軽量。
- LLM-jp 32b: 320 億のうち 3B だけ活性化する MoE 構造、日本語特化、推論力高め。
- Qwen 27b: Alibaba 製の多言語モデル、思考が緻密。
- Gemma 31b: Google 製のオープン重み、対話最適化、表現豊か。
"""

def prompt_for(self_name):
    return (
        "あなたは AI 言語モデルです。以下 4 つのモデルを、それぞれ **絵文字 5 つ** だけで表現してください。\n"
        "自分自身を含みます。説明文や形容詞は禁止、絵文字のみ。\n\n"
        f"あなた自身: {self_name}\n\n"
        f"モデル一覧と説明:\n{MODEL_HINTS}\n"
        "出力形式は **厳密に** 以下:\n"
        "LLM-jp 8b: 🅰️🅱️🅲️🅳️🅴️\n"
        "LLM-jp 32b: 🅰️🅱️🅲️🅳️🅴️\n"
        "Qwen 27b: 🅰️🅱️🅲️🅳️🅴️\n"
        "Gemma 31b: 🅰️🅱️🅲️🅳️🅴️\n\n"
        "(🅰️〜🅴️ は実際の絵文字に置き換えてください)"
    )

# 行 = describer (誰が描いたか)、列 = subject (誰について)
SUBJECT_KEYS = ["LLM-jp 8b", "LLM-jp 32b", "Qwen 27b", "Gemma 31b"]

# 「LLM-jp 8b:」のような行から絵文字を抽出する
LINE_PAT = re.compile(r"^[-\*\s]*([A-Za-z0-9 \-]+?)\s*[:：]\s*(.+)$")

def parse_emoji_map(text):
    out = {}
    for line in text.splitlines():
        m = LINE_PAT.match(line)
        if not m: continue
        key = m.group(1).strip()
        emojis = m.group(2).strip()
        # 各 SUBJECT_KEYS に対して部分一致
        for sk in SUBJECT_KEYS:
            if sk.lower() in key.lower():
                out[sk] = emojis
                break
    return out

matrix = {}  # describer_voice_name -> {subject_key -> emojis}
for describer_name, model_id in VOICES.items():
    # describer は VOICES の中の名前 (🌸 付き)。プロンプトでは絵文字なし版を渡す
    plain = describer_name.split(" ", 1)[-1]  # "LLM-jp 8b"
    print(f"  🎨 {describer_name} に依頼中 ...", end=" ", flush=True)
    raw = chat(model_id, prompt_for(plain), temperature=0.7, max_tokens=4000)
    parsed = parse_emoji_map(raw)
    matrix[describer_name] = parsed
    print(f"パース成功 {len(parsed)}/4")
    for k in SUBJECT_KEYS:
        print(f"     {k}: {parsed.get(k, '(failed)')}")


## 3. 4×4 マトリックス表示

In [ ]:
# 行 = describer、列 = subject、対角が「自画像」
headers = ["**描く側 ↓ ／ 描かれる側 →**"] + [f"**{s}**" for s in SUBJECT_KEYS]
rows = []
for describer_name in VOICES:
    row = [f"**{describer_name}**"]
    for s in SUBJECT_KEYS:
        cell = matrix.get(describer_name, {}).get(s, "─")
        plain_self = describer_name.split(" ", 1)[-1]
        if plain_self == s:
            cell = f"🪞 {cell}"  # 自画像を強調
        row.append(cell)
    rows.append(row)

table = (
    "| " + " | ".join(headers) + " |\n"
    "|" + "|".join(["---"] * len(headers)) + "|\n" +
    "\n".join("| " + " | ".join(r) + " |" for r in rows)
)
display(Markdown(f"## 🎭 絵文字マトリックス (🪞 = 自画像)\n\n{table}"))


## 4. 自己 vs 他者ギャップを観察

In [ ]:
# 各 subject について、「自画像」とそれ以外 3 つの「他者描写」を並べる
display(Markdown("## 🪞 自画像 vs 👀 他者の見方"))
for s in SUBJECT_KEYS:
    self_voice = next((v for v in VOICES if v.split(" ", 1)[-1] == s), None)
    self_view = matrix.get(self_voice, {}).get(s, "─")
    others = []
    for describer_name in VOICES:
        if describer_name == self_voice: continue
        others.append(f"- {describer_name}: {matrix.get(describer_name, {}).get(s, '─')}")
    display(Markdown(
        f"### {s}\n\n"
        f"**🪞 自画像**: {self_view}\n\n"
        f"**👀 他モデルから見た {s}**:\n" + "\n".join(others)
    ))


## おまけ

- `MODEL_HINTS` を空にして再実行 — モデルが名前のみから連想するとどう変わるか
- 結果をスクショして社内 Slack で共有すると盛り上がります
- 教訓: モデル自身の「自己認識」はわりと無難で、他者描写の方が大胆になる傾向あり (人間と同じ？)
- `temperature` を変えると絵文字選択の自由度が変わります。`0.0` でやると保守的、`1.2` でやると突飛
